# 01 · Data collection — Sentinel-1 patches with WorldCover labels

Builds the training / validation / test patch datasets from Google Earth Engine.

| Item | Source |
|---|---|
| Input | `COPERNICUS/S1_GRD`, IW mode, VV + VH (dB), monthly median, one consistent relative orbit per patch |
| Labels | `ESA/WorldCover/v200` (2021), remapped to 6 classes; shrubland, snow, wetland, mangrove, moss → ignore (255) |
| Grid | 10 m, local UTM zone, 128 × 128 px patches (1.28 km) |
| Period | 12 months: Oct 2025 – Sep 2026 |
| Split | spatial blocks (no patch location appears in two splits) |

Output files (`data/patches/`): `train.npz`, `val.npz`, `test.npz`, each with
`X` (N, 2, 128, 128, normalised), `y` (N, 128, 128), `class_medians`, `class_iqr`, `metadata`.
One sample = one location in one month, so N = locations × 12.

Requires an Earth Engine–enabled Google Cloud project (`EE_PROJECT`).

In [ ]:
import os, sys, json, time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ee

PROJECT_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(PROJECT_DIR))

from src import gee
from src.classes import CLASS_NAMES, CLASS_COLORS, CLASS_IDS, IGNORE_INDEX
from src.location import utm_epsg_for

gee.initialize(os.environ.get("EE_PROJECT"))   # or gee.initialize("your-project-id")
print("Project dir:", PROJECT_DIR)
print("Earth Engine:", ee.__version__)

## 1. Configuration

In [ ]:
CENTER_LAT, CENTER_LON = 48.7758, 9.1829    # Stuttgart
CORE_KM   = 20      # inner square: source of validation + test blocks and some training
OUTER_KM  = 40      # outer square: extra training locations (the "V5 outer ring")
PATCH     = 128     # pixels
PIXEL_M   = 10
BLOCK_PATCHES = 3   # spatial block = 3 x 3 patches (3.84 km) for the split
VAL_FRAC, TEST_FRAC = 0.15, 0.15
N_OUTER   = 600     # outer-ring training locations to sample
START_MONTH = "2025-10"
SEED = 42

SCHEDULE = gee.month_schedule(START_MONTH)
MONTHS = [m["label"].replace("-", "_") for m in SCHEDULE]

DATA_DIR = PROJECT_DIR / "data"
RAW_DIR = DATA_DIR / "raw_patches"
OUT_DIR = DATA_DIR / "patches"
RAW_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

EPSG = utm_epsg_for(CENTER_LAT, CENTER_LON)
from pyproj import Transformer
cx, cy = Transformer.from_crs("EPSG:4326", EPSG, always_xy=True).transform(CENTER_LON, CENTER_LAT)
print("UTM:", EPSG, "| centre:", round(cx), round(cy), "| months:", MONTHS[0], "→", MONTHS[-1])

## 2. Candidate patch locations and spatial split

In [ ]:
rng = np.random.default_rng(SEED)
size_m = PATCH * PIXEL_M

def patch_grid(extent_km):
    n = int(extent_km * 1000 // size_m)
    x0 = round((cx - n * size_m / 2) / PIXEL_M) * PIXEL_M
    y0 = round((cy + n * size_m / 2) / PIXEL_M) * PIXEL_M
    return [(r, c, x0 + c * size_m, y0 - r * size_m) for r in range(n) for c in range(n)], n

core, n_core = patch_grid(CORE_KM)
outer_all, n_outer = patch_grid(OUTER_KM)

# Core patches -> blocks -> train / val / test
blocks = {}
for r, c, x, y in core:
    blocks.setdefault((r // BLOCK_PATCHES, c // BLOCK_PATCHES), []).append((x, y))
block_ids = list(blocks)
rng.shuffle(block_ids)
n_val = round(len(block_ids) * VAL_FRAC)
n_test = round(len(block_ids) * TEST_FRAC)
split_of_block = {b: "val" for b in block_ids[:n_val]}
split_of_block.update({b: "test" for b in block_ids[n_val:n_val + n_test]})

locations = []
for b, pts in blocks.items():
    split = split_of_block.get(b, "train")
    # keep one patch per val/test block centre-ish to limit near-duplicates
    chosen = pts if split == "train" else [pts[len(pts) // 2]]
    locations += [{"x": x, "y": y, "split": split, "ring": "core"} for x, y in chosen]

# Outer ring: patches outside the core square, training only
core_set = {(x, y) for _, _, x, y in core}
half_core = CORE_KM * 1000 / 2 + size_m   # 1-patch buffer around the core
ring = [(x, y) for _, _, x, y in outer_all
        if (x, y) not in core_set
        and (abs(x + size_m / 2 - cx) > half_core or abs(y - size_m / 2 - cy) > half_core)]
idx = rng.choice(len(ring), size=min(N_OUTER, len(ring)), replace=False)
locations += [{"x": ring[i][0], "y": ring[i][1], "split": "train", "ring": "outer"} for i in idx]

locations = pd.DataFrame(locations)
locations["location_id"] = np.arange(len(locations))
print(locations.groupby(["split", "ring"]).size())

fig, ax = plt.subplots(figsize=(6, 6))
for split, color in [("train", "#7f8c8d"), ("val", "#2980b9"), ("test", "#c0392b")]:
    s = locations[locations.split == split]
    ax.scatter(s.x / 1000, s.y / 1000, s=6, c=color, label=f"{split} ({len(s)})", marker="s")
ax.set_aspect("equal"); ax.legend(); ax.set_xlabel("Easting (km)"); ax.set_ylabel("Northing (km)")
ax.set_title("Patch locations by split")
plt.show()

## 3. Download patches

One request per location returns all 12 monthly VV/VH composites plus the label band
(25 bands, 128 × 128) via `ee.data.computePixels`. Raw dB values are cached in
`data/raw_patches/` so the cell can be re-run after interruptions.

In [ ]:
def fetch_location(row):
    path = RAW_DIR / f"loc_{row.location_id:05d}.npz"
    if path.exists():
        return row.location_id, "cached"

    grid = {"crs": f"EPSG:{EPSG}", "transform": [PIXEL_M, 0, row.x, 0, -PIXEL_M, row.y],
            "bounds": (row.x, row.y - size_m, row.x + size_m, row.y),
            "width": PATCH, "height": PATCH}
    region = gee.ee_region(grid)
    acq = gee.select_acquisition(region, SCHEDULE)

    bands = []
    for m, label in zip(SCHEDULE, MONTHS):
        bands.append(gee.monthly_composite(region, m, acq).rename([f"VV_{label}", f"VH_{label}"]))
    image = ee.Image.cat(bands + [gee.worldcover_labels(region).rename("label")])

    arr = ee.data.computePixels({
        "expression": image,
        "fileFormat": "NUMPY_NDARRAY",
        "grid": {
            "dimensions": {"width": PATCH, "height": PATCH},
            "affineTransform": {"scaleX": PIXEL_M, "shearX": 0, "translateX": row.x,
                                "shearY": 0, "scaleY": -PIXEL_M, "translateY": row.y},
            "crsCode": f"EPSG:{EPSG}",
        },
    })
    radar = np.stack([np.stack([arr[f"VV_{l}"], arr[f"VH_{l}"]]) for l in MONTHS]).astype(np.float32)
    np.savez_compressed(path, radar=radar, label=arr["label"].astype(np.uint8),
                        orbit_pass=acq["orbit_pass"], relative_orbit=acq["relative_orbit"] or -1)
    return row.location_id, acq["note"]

t0, done, failed = time.time(), 0, []
with ThreadPoolExecutor(max_workers=8) as pool:
    futures = {pool.submit(fetch_location, r): r.location_id for r in locations.itertuples()}
    for f in as_completed(futures):
        try:
            f.result(); done += 1
        except Exception as exc:
            failed.append((futures[f], str(exc)[:120]))
        if (done + len(failed)) % 100 == 0:
            print(f"{done + len(failed)}/{len(locations)}  ({time.time() - t0:.0f}s)")

print(f"Downloaded: {done}  Failed: {len(failed)}  Time: {(time.time() - t0) / 60:.1f} min")
for fid, err in failed[:10]:
    print("  failed", fid, err)

## 4. Quality control

In [ ]:
NODATA = gee.NODATA
records = []
for row in locations.itertuples():
    path = RAW_DIR / f"loc_{row.location_id:05d}.npz"
    if not path.exists():
        continue
    with np.load(path) as d:
        radar, label = d["radar"], d["label"]
    invalid = ((radar == NODATA) | ~np.isfinite(radar)).any(axis=(0, 1)).mean()
    labelled = (label != IGNORE_INDEX).mean()
    records.append({"location_id": row.location_id, "invalid_frac": invalid, "labelled_frac": labelled})

qc = locations.merge(pd.DataFrame(records), on="location_id")
keep = qc[(qc.invalid_frac <= 0.01) & (qc.labelled_frac >= 0.5)].copy()
print(f"Locations kept: {len(keep)} / {len(qc)}")
print(keep.groupby("split").size())

## 5. Normalisation statistics (training split only)

In [ ]:
sample = []
for lid in keep[keep.split == "train"].location_id.sample(min(200, (keep.split == "train").sum()), random_state=SEED):
    with np.load(RAW_DIR / f"loc_{lid:05d}.npz") as d:
        r = d["radar"]
    sample.append(r[:, :, ::4, ::4].transpose(1, 0, 2, 3).reshape(2, -1))
sample = np.concatenate(sample, axis=1)
sample = np.where(sample == NODATA, np.nan, sample)

class_medians = np.nanmedian(sample, axis=1).astype(np.float32)
q75, q25 = np.nanpercentile(sample, [75, 25], axis=1)
class_iqr = (q75 - q25).astype(np.float32)
print("VV / VH median (dB):", class_medians)
print("VV / VH IQR    (dB):", class_iqr)

## 6. Build and save the split files

In [ ]:
meta_dtype = [("location_id", "i4"), ("month", "U7"), ("x_min", "f8"), ("y_max", "f8"), ("ring", "U5")]

def build(split):
    rows = keep[keep.split == split]
    X = np.empty((len(rows) * 12, 2, PATCH, PATCH), np.float32)
    y = np.empty((len(rows) * 12, PATCH, PATCH), np.uint8)
    meta = np.empty(len(rows) * 12, dtype=meta_dtype)
    i = 0
    for row in rows.itertuples():
        with np.load(RAW_DIR / f"loc_{row.location_id:05d}.npz") as d:
            radar, label = d["radar"], d["label"]
        for m, month in enumerate(MONTHS):
            r = radar[m].copy()
            for b in range(2):
                bad = (r[b] == NODATA) | ~np.isfinite(r[b])
                r[b][bad] = class_medians[b]
            X[i] = (r - class_medians[:, None, None]) / class_iqr[:, None, None]
            y[i] = label
            meta[i] = (row.location_id, month, row.x, row.y, row.ring)
            i += 1
    path = OUT_DIR / f"{split}.npz"
    np.savez_compressed(path, X=X, y=y, metadata=meta,
                        class_medians=class_medians, class_iqr=class_iqr)
    return path, X.shape, y

summary = {}
for split in ["train", "val", "test"]:
    path, shape, y = build(split)
    counts = {CLASS_NAMES[c]: int((y == c).sum()) for c in CLASS_IDS}
    counts["Ignore"] = int((y == IGNORE_INDEX).sum())
    summary[split] = counts
    print(f"{split:5s} {shape}  ->  {path.relative_to(PROJECT_DIR)}  ({path.stat().st_size / 1e6:.0f} MB)")

dist = pd.DataFrame(summary)
display((dist / dist.sum() * 100).round(2).rename(columns=lambda c: f"{c} (%)"))

## 7. Class distribution and sample patches

In [ ]:
share = (dist.drop(index="Ignore") / dist.drop(index="Ignore").sum() * 100)
ax = share.plot.bar(figsize=(10, 4), color=["#7f8c8d", "#2980b9", "#c0392b"])
ax.set_ylabel("Share of labelled pixels (%)"); ax.set_title("Class distribution per split")
plt.xticks(rotation=20, ha="right"); plt.tight_layout(); plt.show()

In [ ]:
from src.visualize import CMAP, NORM, LEGEND

with np.load(OUT_DIR / "val.npz") as d:
    Xv, yv, mv = d["X"], d["y"], d["metadata"]

picks = np.linspace(0, len(Xv) - 1, 4).astype(int)
fig, axes = plt.subplots(3, 4, figsize=(13, 10))
for j, i in enumerate(picks):
    axes[0, j].imshow(Xv[i, 0], cmap="gray", vmin=-3, vmax=3); axes[0, j].set_title(f"VV  loc {mv[i]['location_id']} {mv[i]['month']}")
    axes[1, j].imshow(Xv[i, 1], cmap="gray", vmin=-3, vmax=3); axes[1, j].set_title("VH")
    lab = np.ma.masked_equal(yv[i], IGNORE_INDEX)
    axes[2, j].imshow(lab, cmap=CMAP, norm=NORM, interpolation="nearest"); axes[2, j].set_title("WorldCover label")
for ax in axes.flat: ax.set_axis_off()
fig.legend(handles=LEGEND, loc="lower center", ncol=6, frameon=False)
plt.tight_layout(rect=(0, 0.04, 1, 1)); plt.show()

## 8. Package the dataset for upload

The archive is too large for GitHub. Upload `data/sentinel1_lulc_dataset.zip` to Zenodo,
Google Drive or Hugging Face Datasets and put the link in the README.
`scripts/package_dataset.py` does the same from the command line.

In [ ]:
import subprocess
subprocess.run([sys.executable, str(PROJECT_DIR / "scripts" / "package_dataset.py")], check=True)